In [1]:
import random

import mlflow
import numpy as np
import pyarrow.parquet as pq
import torch.cuda
from torch import nn
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 128
EPOCHS = 30
LR = 1e-3
NUM_CLASSES = 8
DATA_DIR = "../data/processed"

print(f"Device: {DEVICE}")

Device: cuda


In [2]:
LABELS = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-full", "Random", "Scratch"]
LABEL2IDX = {name: i for i, name in enumerate(LABELS)}


class WaferDataset(Dataset):
    def __init__(self, parquet_path: str):
        table = pq.read_table(parquet_path)
        df = table.to_pandas()
        wafers = np.stack(df["wafer"].values)
        self.X = wafers.reshape(-1, 1, 64, 64).astype(np.float32)
        self.y = df["label"].map(LABEL2IDX).values.astype(np.int64)
        assert not np.isnan(self.y).any(), "Unknown label found"

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), int(self.y[idx])


train_ds = WaferDataset(f"{DATA_DIR}/train.parquet")
val_ds = WaferDataset(f"{DATA_DIR}/val.parquet")
test_ds = WaferDataset(f"{DATA_DIR}/test.parquet")

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"X shape={train_ds.X.shape}, dtype={train_ds.X.dtype}")


Train: 17863, Val: 3828, Test: 3828
X shape=(17863, 1, 64, 64), dtype=float32


In [3]:
# count samples per classes in the training set
class_counts = np.bincount(train_ds.y, minlength=NUM_CLASSES)
for i, name in enumerate(LABELS):
    print(f"{name:12s}: {class_counts[i]}")

# Weight per class = inverse frequency -> rare classes get higher weight
class_weights = 1.0 / class_counts
# Assign each sample the weight of its class
sample_weights = class_weights[train_ds.y]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).double(),  # sampler requires float64
    num_samples=len(sample_weights),  # keep one epoch ~= dataset size
    replacement=True
)

print(f"Sampler ready, {len(sample_weights)} samples")

Center      : 3006
Donut       : 389
Edge-Loc    : 3632
Edge-Ring   : 6776
Loc         : 2515
Near-full   : 104
Random      : 606
Scratch     : 835
Sampler ready, 17863 samples


In [4]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH,
    sampler=sampler,  # weighted sampling -> no shuffle (sample controls order)
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH,
    shuffle=False,  # natural distribution, order irrelevant for eval
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Sanity check
xb, yb = next(iter(train_loader))
print(f"batch X: {xb.shape}  {yb.dtype}")  # expect [128, 1, 64, 64] float32
print(f"batch y: {yb.shape}  {yb.dtype}")  # expect [128] int64
print(f"class count in this batch: {np.bincount(yb.numpy(), minlength=NUM_CLASSES)}")

batch X: torch.Size([128, 1, 64, 64])  torch.int64
batch y: torch.Size([128])  torch.int64
class count in this batch: [13 22 17 15 15 14 10 22]


In [5]:
class WaferCNN(nn.Module):
    """Small baseline CNN for 64x64 single channel wafer maps."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # block 1: 1-> 32, 64x64 -> 32 x 32
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # block 2: 32 -> 64, 32 x 32 -> 16 x 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # block 3: 64 -> 128, 16 x 16 -> 8 x 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # 128x8x8 -> 128x1x1, robust to size changes
            nn.Flatten(),  # -> [N, 128]
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = x / 2.0  # normalize die values {0, 1, 2} -> {0, 0.5, 1.0}
        x = self.features(x)
        x = self.classifier(x)
        return x


model = WaferCNN().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"trainable params: {n_params:,}")

for name, p in model.named_parameters():
    print(f"{name:35s} {str(type(p.shape)):20s} {p.numel():>8,}")

# Forward sanity check on each batch
with torch.no_grad():
    out = model(xb.to(DEVICE))
print(f"output shape: {out.shape}")  # expect [128,8]

WaferCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim

In [6]:
# γ và β init of BatchNorm — weight = γ, bias = β
bn = model.features[1]  # BatchNorm2d(32)
print("γ (weight):", bn.weight.data[:5])  # all 1.0
print("β (bias):  ", bn.bias.data[:5])  # all 0.0

# He-init randomly
conv = model.features[0]  # Conv2d(1, 32)
print("conv weight:", conv.weight.data[0, 0])  # 3×3 random small number

γ (weight): tensor([1., 1., 1., 1., 1.], device='cuda:0')
β (bias):   tensor([0., 0., 0., 0., 0.], device='cuda:0')
conv weight: tensor([[ 0.2048,  0.1338,  0.2557],
        [-0.2877,  0.2651, -0.0399],
        [-0.0570,  0.1748,  0.0881]], device='cuda:0')


In [7]:
from pathlib import Path

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam, a solid default for CNNs
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Mlflow: point to a local tracking dir and name the experiment
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
db_path = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("wm811k-baseline-cnn")

print("criterion, optimizer, and experiment set up!")

criterion, optimizer, and experiment set up!
